In [1]:
AXIS_CHANNELS = {
    "x": [0, 3, 6, 9, 12, 15],
    "y": [1, 4, 7, 10, 13, 16],
    "z": [2, 5, 8, 11, 14, 17]
}

SIGNAL_CHANNELS = {
    "accelerometer": [0, 1, 2],
    "gravity": [3, 4, 5],
    "gyros": [6, 7, 8],
    "lin_accel": [9, 10, 11],
    "game_rot_vec": [12, 13, 14],
    "magn_field": [15, 16, 17]
}

## XROCKET Encoder & Predictions

In [2]:
import sys
import ast
import numpy as np

from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.feature_selection import f_classif

from torch import Tensor, concat
from utils import input_generation, dilation_combinations, channel_combinations


# generate combinations of channels
ablation_experiments = {"baseline": None, **channel_combinations(AXIS_CHANNELS), **channel_combinations(SIGNAL_CHANNELS)}


prediction_results_1_1 = []

channel_results = []
dilation_results = []
pattern_results = []

#------------------------------
# SIGNAL EXPERIMENTS (TASK 1.1)
#------------------------------
for experiment, channels_to_remove in ablation_experiments.items():

    X, y, channel_names = input_generation(Path("KSAS-Dataset/movements"), channels_to_remove)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


    #-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

    #-----------------
    # X-ROCKET ENCODER
    #-----------------
    repo_path = str(Path.cwd() / "xrocket")
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    from xrocket.encoder import XRocket

    encoder = XRocket(in_channels=len(channel_names), max_kernel_span=33, combination_order=1, feature_cap=10_000, kernel_length=9, max_dilations=32)
    encoder.fit(Tensor(X_train[0]).unsqueeze(0))


    #-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

    embed_train, embed_test = [], []

    for x in X_train:
        embed_train.append(encoder(Tensor(x).unsqueeze(0)))

    for x in X_test:
        embed_test.append(encoder(Tensor(x).unsqueeze(0)))


    feature_names = pd.Index(encoder.feature_names).rename(["pattern", "dilation", "channel", "threshold"])
    embed_train = pd.DataFrame(data=concat(embed_train), columns=feature_names)
    embed_test = pd.DataFrame(data=concat(embed_test), columns=feature_names)


    #-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

    #----------------------------------
    # DILATION EXPERIMENTS (TASK 1.2)
    #----------------------------------
    if channels_to_remove is None:


        # Baseline with all dilations
        dilation_experiments = {"baseline": None, **dilation_combinations(list(encoder.dilations))}

        for name, dilation in dilation_experiments.items():

            if dilation is None:
                train_dilation = embed_train
                test_dilation = embed_test

            else:
                # Remove features belonging to these dilations
                dilation_mask = ~embed_train.columns.get_level_values("dilation").isin(dilation)

                train_dilation = embed_train.loc[:, dilation_mask]
                test_dilation = embed_test.loc[:, dilation_mask]


            #-----------------------
            # PREDICTIONS (TASK 1.2)
            #-----------------------
            dilation_model = RandomForestClassifier(random_state=42)
            dilation_model.fit(train_dilation, y_train)

            dilation_pred = dilation_model.predict(test_dilation)

            dilation_accuracy = accuracy_score(y_test, dilation_pred)
            dilation_precision = precision_score(y_test, dilation_pred, average="macro", zero_division=0)
            dilation_recall = recall_score(y_test, dilation_pred, average="macro", zero_division=0)

            dilation_results.append({
                "experiment": name,
                "removed_dilations": dilation,
                "remaining_features": train_dilation.shape[1],
                "accuracy": dilation_accuracy,
                "precision": dilation_precision,
                "recall": dilation_recall
            })


    #-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

    #-----------------------
    # PREDICTIONS (TASK 1.1)
    #-----------------------

    model = RandomForestClassifier(random_state=42)
    model.fit(embed_train, y_train)

    pred_test = model.predict(embed_test)

    for true_label, predicted_label in zip(y_test, pred_test):

        prediction_results_1_1.append({
            "experiment": experiment,
            "true_label": true_label,
            "predicted_label": predicted_label
        })


    #-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

    #--------------------------------
    # PATTERN ANALYSIS (TASK 1.3)
    #--------------------------------
    if channels_to_remove is None:

        # Analyze each movement against all the others
        for movement in sorted(set(y_train)):

            # One-vs-rest labels
            y_binary = (y_train == movement).astype(int)

            # ANOVA F-score for every XROCKET feature
            f_scores, p_values = f_classif(embed_train, y_binary)

            # Select the 20 most discriminative patterns for this movement
            top_feature_positions = np.argsort(f_scores)[::-1][:20]

            for position in top_feature_positions:

                # Decode channel
                channel_vector = ast.literal_eval(embed_train.columns[position][2])
                channel_index = int(np.argmax(channel_vector))
                channel = channel_names[channel_index]

                # Mean activation for the movement and for all the other movements
                movement_mean = embed_train.iloc[y_binary == 1, position].mean()
                rest_mean = embed_train.iloc[y_binary == 0, position].mean()

                pattern_results.append({
                    "movement": movement,
                    "feature_position": position,
                    "pattern": embed_train.columns[position][0],
                    "dilation": embed_train.columns[position][1],
                    "channel": channel,
                    "threshold": embed_train.columns[position][3],
                    "f_score": f_scores[position],
                    "p_value": p_values[position],
                    "movement_mean": movement_mean,
                    "rest_mean": rest_mean
                })


        pattern_results_df = pd.DataFrame(pattern_results)

        Path("results").mkdir(exist_ok=True)
        pattern_results_df.to_csv("results/task_1_3_patterns.csv", index=False)

        print(pattern_results_df)


    #-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

    # Metrics for each movement (Task 1.1)
    for movement in sorted(set(y_test)):

        y_true_binary = (y_test == movement).astype(int)
        y_pred_binary = (pred_test == movement).astype(int)

        movement_precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
        movement_recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)

        channel_results.append({
            "experiment": experiment,
            "removed_channels": 0 if channels_to_remove is None else len(channels_to_remove),
            "remaining_channels": len(channel_names),
            "movement": movement,
            "precision": movement_precision,
            "recall": movement_recall
        })

     movement  feature_position  \
0           0              2172   
1           0              3689   
2           0              2459   
3           0              4009   
4           0              3692   
..        ...               ...   
115         5              5033   
116         5              4919   
117         5              3408   
118         5              4906   
119         5              3423   

                                               pattern  dilation  \
0    [-1.0, -1.0, -1.0, 2.0, -1.0, -1.0, 2.0, -1.0,...         2   
1    [-1.0, -1.0, -1.0, -1.0, 2.0, -1.0, 2.0, 2.0, ...         3   
2    [2.0, -1.0, -1.0, -1.0, -1.0, 2.0, -1.0, 2.0, ...         2   
3    [-1.0, -1.0, 2.0, -1.0, -1.0, -1.0, 2.0, 2.0, ...         3   
4    [-1.0, -1.0, -1.0, -1.0, -1.0, 2.0, 2.0, 2.0, ...         3   
..                                                 ...       ...   
115  [-1.0, -1.0, -1.0, -1.0, 2.0, -1.0, 2.0, 2.0, ...         4   
116  [-1.0, 2.0, -1.0, -1.0, -1.0, 

## RESULTS VISUALIZATION

# Task 1.1

In [3]:
prediction_results_df = pd.DataFrame(prediction_results_1_1)

prediction_results_df.to_csv("experimental_results/task_1_1/task_1_1_predictions.csv", index=False)

In [4]:
results_df = pd.DataFrame(channel_results)


#----------------
# RECALL TABLE
#----------------
recall_table = results_df.pivot(index="experiment", columns="movement", values="recall")

recall_table.to_csv("experimental_results/task_1_1/recall.csv")

recall_table


movement,0,1,2,3,4,5
experiment,,,,,,
baseline,1.0,0.916667,0.750000,0.750000,0.833333,1.000000
remove_accelerometer,1.0,0.916667,0.750000,0.583333,0.833333,0.916667
remove_accelerometer_game_rot_vec,1.0,1.000000,0.833333,0.583333,0.833333,1.000000
remove_accelerometer_game_rot_vec_magn_field,1.0,1.000000,0.750000,0.583333,0.833333,1.000000
remove_accelerometer_gravity,1.0,0.916667,0.416667,0.500000,0.833333,1.000000
...,...,...,...,...,...,...
remove_x_y,1.0,1.000000,0.666667,0.666667,0.583333,0.833333
remove_x_z,1.0,0.916667,0.666667,0.416667,0.833333,1.000000
remove_y,1.0,1.000000,0.750000,0.583333,0.666667,0.916667


In [5]:
#-------------------
# PRECISION TABLE
#-------------------
precision_table = results_df.pivot(index="experiment", columns="movement", values="precision")

precision_table.to_csv("experimental_results/task_1_1/precision.csv")

precision_table

movement,0,1,2,3,4,5
experiment,,,,,,
baseline,1.0,0.916667,0.818182,0.900000,0.833333,0.800000
remove_accelerometer,1.0,0.846154,0.692308,0.875000,0.833333,0.785714
remove_accelerometer_game_rot_vec,1.0,0.923077,0.714286,1.000000,0.909091,0.800000
remove_accelerometer_game_rot_vec_magn_field,1.0,0.923077,0.692308,0.875000,0.909091,0.800000
remove_accelerometer_gravity,1.0,1.000000,0.625000,0.600000,0.714286,0.705882
...,...,...,...,...,...,...
remove_x_y,1.0,0.857143,0.571429,0.666667,1.000000,0.769231
remove_x_z,1.0,0.916667,0.666667,0.714286,0.769231,0.750000
remove_y,1.0,0.857143,0.642857,0.777778,0.888889,0.785714


# Task 1.2

In [6]:
dilation_results_df = pd.DataFrame(dilation_results)

# Use experiment names as index
accuracy_table = dilation_results_df.set_index("experiment")

# Baseline accuracy
baseline = accuracy_table.loc["baseline", "accuracy"]

accuracy_table.to_csv("experimental_results/task_1_2_accuracy.csv")

accuracy_table

,removed_dilations,remaining_features,accuracy,precision,recall
experiment,,,,,
baseline,None,6048,0.875000,0.878030,0.875000
remove_1,[1],4536,0.819444,0.819231,0.819444
remove_2,[2],4536,0.833333,0.832407,0.833333
remove_3,[3],4536,0.847222,0.850748,0.847222
remove_4,[4],4536,0.819444,0.811905,0.819444
remove_1_2,"[1, 2]",3024,0.791667,0.792277,0.791667
remove_1_3,"[1, 3]",3024,0.847222,0.852885,0.847222
remove_1_4,"[1, 4]",3024,0.847222,0.852885,0.847222
remove_2_3,"[2, 3]",3024,0.847222,0.852885,0.847222


In [7]:
dilation_results_df = pd.DataFrame(dilation_results)

# Use experiment names as index
metrics_table = dilation_results_df.set_index("experiment")

# Baseline metrics
baseline_accuracy = metrics_table.loc["baseline", "accuracy"]
baseline_precision = metrics_table.loc["baseline", "precision"]
baseline_recall = metrics_table.loc["baseline", "recall"]

metrics_table.to_csv("experimental_results/task_1_2_metrics.csv")

metrics_table

,removed_dilations,remaining_features,accuracy,precision,recall
experiment,,,,,
baseline,None,6048,0.875000,0.878030,0.875000
remove_1,[1],4536,0.819444,0.819231,0.819444
remove_2,[2],4536,0.833333,0.832407,0.833333
remove_3,[3],4536,0.847222,0.850748,0.847222
remove_4,[4],4536,0.819444,0.811905,0.819444
remove_1_2,"[1, 2]",3024,0.791667,0.792277,0.791667
remove_1_3,"[1, 3]",3024,0.847222,0.852885,0.847222
remove_1_4,"[1, 4]",3024,0.847222,0.852885,0.847222
remove_2_3,"[2, 3]",3024,0.847222,0.852885,0.847222


# Task 1.3

In [8]:
pattern_table = (pattern_results_df[["movement", "feature_position", "channel", "dilation", "f_score", "p_value", "movement_mean", "rest_mean"]].sort_values(["movement", "f_score"], ascending=[True, False]).reset_index(drop=True))

pattern_table.to_csv("experimental_results/task_1_3_discriminative_patterns.csv", index=False)

pattern_table

,movement,feature_position,channel,dilation,f_score,p_value,movement_mean,rest_mean
0,0,2172,game_rot_vec_x,2,338.961731,5.956920e-42,0.148847,0.487774
1,0,3689,magn_field_z,3,310.347107,7.659778e-40,0.130559,0.496714
2,0,2459,lin_accel_z,2,308.352325,1.086334e-39,0.213000,0.491592
3,0,4009,game_rot_vec_y,3,304.695587,2.069247e-39,0.149657,0.464226
4,0,3692,accelerometer_z,3,300.920288,4.046098e-39,0.129061,0.496973
...,...,...,...,...,...,...,...,...
115,5,5033,lin_accel_z,4,68.415031,4.099814e-14,0.463571,0.188136
116,5,4919,gravity_z,4,66.117149,9.395783e-14,0.560196,0.323850
117,5,3408,gyros_x,3,66.079315,9.525643e-14,0.511798,0.285469
118,5,4906,lin_accel_y,4,65.575592,1.143828e-13,0.269017,0.510837
